# Week 2 Extension: Learning the Environment-Designer MDP

The first Week 2 notebook used declared transition probabilities. This extension retains the same states, actions, Bellman evaluation, and policy-iteration code, but estimates $P(s'\mid s,a)$ from Week 1 history and new action-conditioned MetaWorld trials.

## Estimator

For observation count $n(s,a,s')$, total $N(s,a)$, hand-model probability $P_0$, and prior strength $\alpha$, the fitted posterior mean is

$$\hat P(s'\mid s,a)=\frac{n(s,a,s')+\alpha P_0(s'\mid s,a)}{N(s,a)+\alpha}.$$

Here $\alpha=1$: the complete hand model supplies one pseudo-observation for cold start. Each real observation includes its random seed, candidate `EnvironmentSpec`, full validation report, destination state, and observed reward.

In [ ]:
import json
from pathlib import Path

from spade_metaworld.empirical_transitions import run_empirical_transition_experiment

project_root = Path.cwd()
if not (project_root / 'pyproject.toml').exists():
    project_root = project_root.parent
history_path = project_root / 'results/example_seed_20260927/history.jsonl'
output_dir = project_root / 'results/week2_empirical_model'
RECOLLECT = False  # Set True to run fresh MetaWorld trials (about 20 seconds).

if RECOLLECT or not (output_dir / 'summary.json').exists():
    result = run_empirical_transition_experiment(
        output_dir, history_path, samples_per_action=3, prior_strength=1.0
    )
    summary = result.as_dict()
else:
    summary = json.loads((output_dir / 'summary.json').read_text())
summary

## Evidence coverage

The checked experiment imports four historical transitions and executes 51 new observations: three trials for every available state–action pair. `accept` is an exact terminal decision; all editing actions construct a mutated `EnvironmentSpec` and pass it through the existing `EnvironmentValidator`.

In [ ]:
from IPython.display import Image, display

display(Image(filename=output_dir / 'transition_evidence.png'))

In [ ]:
model = json.loads((output_dir / 'empirical_transition_model.json').read_text())
print('state          action                 next state      count probability reward')
for row in model:
    print(
        f"{row['state']:14s} {row['action']:22s} {row['next_state']:14s} "
        f"{row['observed_count']:5d} {row['probability']:11.3f} {row['reward_mean']:7.3f}"
    )

## Policy change

The empirical policy still repairs invalid proposals and accepts only a target environment. The important change is at `varied` and `intermediate`: observed `advance_family` operations reliably moved to the next complexity band, while `tighten_constraints` normally remained in the same band. Policy iteration therefore replaces the hand model's `tighten_constraints` choices with `advance_family`.

In [ ]:
print('state          hand model             empirical model')
for state in summary['empirical_policy']:
    print(
        f"{state:14s} {summary['manual_policy'][state]!s:22s} "
        f"{summary['empirical_policy'][state]}"
    )

## Limits and next data-collection decision

All 51 active samples passed basic MetaWorld validation. Consequently, this model estimates environment-construction reliability and complexity movement—not learned robot-policy success. The next scientifically useful data should come from more seeds and from an actual trained/evaluated control policy. Confidence intervals should also be reported before using small probability differences operationally.